# Homework 1 — Introduction to Machine Learning · ML Zoomcamp 2026

**Dataset:** `car_fuel_efficiency_2026.csv` — release *pinned* del repositorio del curso
(`cohorts/2026/data/`).

Preguntas **Q1–Q7**. Al final: **resumen de respuestas** (para copiar en el formulario de
entrega) y la sección de *Learning in Public*.

Formulario: <https://courses.datatalks.club/ml-zoomcamp-2026/homework/hw01>

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

## Q1. Pandas version

¿Qué versión de Pandas instalaste?

```python
pd.__version__
```

In [2]:
print("Respuesta Q1:", pd.__version__)

Respuesta Q1: 3.0.6


## Load the dataset

Usamos la copia *pinned* del repo del curso si existe (offline y determinista);
si no, caemos a la URL oficial del homework.

In [3]:
DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
CANDIDATES = [
    Path("cohorts/2026/data/car_fuel_efficiency_2026.csv"),  # cwd = raiz del repo
    Path("../../cohorts/2026/data/car_fuel_efficiency_2026.csv"),  # cwd = 01-intro/homework1
    Path("machine-learning-zoomcamp/cohorts/2026/data/car_fuel_efficiency_2026.csv"),  # cwd = c:/mlzoomcamp
]
_local = next((p for p in CANDIDATES if p.exists()), None)
source = _local if _local is not None else DATA_URL

df = pd.read_csv(source)
print("Fuente:", source)
print("Shape:", df.shape)
df.head()

Fuente: ..\..\cohorts\2026\data\car_fuel_efficiency_2026.csv
Shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Q2. Records count

¿Cuántos registros hay en el dataset?

- 5000
- 9000
- 10000
- 15000

In [4]:
n_records = len(df)
print("Respuesta Q2:", n_records)

Respuesta Q2: 10000


## Q3. Fuel types

¿Cuántos tipos de combustible hay en el dataset?

- 1
- 2
- 3
- 4

In [5]:
fuel_types = df["fuel_type"].unique()
print("Valores:", sorted(fuel_types))
print("Respuesta Q3:", df["fuel_type"].nunique())

Valores: ['Diesel', 'Gasoline', 'Hybrid']
Respuesta Q3: 3


## Q4. Missing values

¿Cuántas columnas del dataset tienen valores faltantes (*missing values*)?

- 0
- 1
- 2
- 3
- 4

In [6]:
missing = df.isna().sum()
missing_cols = missing[missing > 0]

print(missing_cols)
n_missing_cols = int((missing > 0).sum())
print("Respuesta Q4:", n_missing_cols)

horsepower      877
acceleration    264
dtype: int64
Respuesta Q4: 2


## Q5. Max fuel efficiency

¿Cuál es la máxima eficiencia de combustible (`fuel_efficiency_mpg`) de los coches de Asia?

- 21.2
- 31.2
- 41.2
- 51.2

In [7]:
asia = df[df["origin"] == "Asia"]
max_mpg_asia = asia["fuel_efficiency_mpg"].max()
print("Maximo global (todos los origenes):", df["fuel_efficiency_mpg"].max())
print("Respuesta Q5:", max_mpg_asia)

Maximo global (todos los origenes): 41.2
Respuesta Q5: 41.2


Nota: hay que **filtrar `origin == "Asia"`** antes de maximizar. El máximo global
resulta ser también `41.2` porque el coche más eficiente es asiático, pero el método
correcto es filtrar primero (si los datos cambian, solo el filtrado sigue dando la
respuesta correcta).

## Q6. Median value of horsepower

1. Encuentra la **mediana** de la columna `horsepower`.
2. Calcula el **valor más frecuente** (*mode*) de la misma columna.
3. Rellena los valores faltantes con `fillna` usando ese mode.
4. Calcula la mediana de nuevo.

¿Ha cambiado la mediana?

- Yes, it increased
- Yes, it decreased
- No

In [8]:
median_before = df["horsepower"].median()
mode_hp = df["horsepower"].mode().iloc[0]

# Rellena los 877 faltantes con el mode (lo hacemos con una copia para no mutar el dataset crudo)
hp_filled = df["horsepower"].fillna(mode_hp)

median_after = hp_filled.median()
n_missing_hp = int(hp_filled.isna().sum())

print("median antes:  ", median_before)
print("mode:          ", mode_hp)
print("median despues:", median_after)
print("faltantes restantes:", n_missing_hp)

if median_after == median_before:
    answer_q6 = "No"
elif median_after > median_before:
    answer_q6 = "Yes, it increased"
else:
    answer_q6 = "Yes, it decreased"

print("Respuesta Q6:", answer_q6)

median antes:   254.0
mode:           252.0
median despues: 252.0
faltantes restantes: 0
Respuesta Q6: Yes, it decreased


La mediana **baja de 254.0 a 252.0**: al rellenar los 877 faltantes (8.77 %) con el
mode (`252.0`, el valor más frecuente) se añaden muchas observaciones en la parte baja
de la distribución, así que la mediana se desplaza hacia abajo → **Yes, it decreased**.

## Q7. Sum of weights

1. Selecciona todos los coches de **Asia**.
2. Selecciona solo las columnas `vehicle_weight` y `model_year`.
3. Toma los **primeros 7** valores.
4. Obtén el array NumPy subyacente → `X`.
5. Calcula `XTX = X.T @ X`.
6. Invierte `XTX`.
7. Crea `y = [1100, 1300, 800, 900, 1000, 1100, 1200]`.
8. Calcula `w = inv(XTX) @ X.T @ y`.
9. ¿Cuál es la **suma de todos los elementos de `w`**?

> Acabas de implementar **regresión lineal** (ecuaciones normales: `XTX · w = X.T · y`).

- 0.0369
- 0.369
- 3.69
- 36.9

In [9]:
asia = df[df["origin"] == "Asia"]
X = asia[["vehicle_weight", "model_year"]].head(7).to_numpy()
X

array([[4240, 1996],
       [4310, 1992],
       [4230, 1997],
       [4420, 1978],
       [3950, 2007],
       [4750, 2021],
       [4450, 2019]])

In [10]:
XTX = X.T.dot(X)  # (2x7) . (7x2) = 2x2 -> invertible
print("XTX =")
print(XTX)
print("det(XTX) =", np.linalg.det(XTX))

XTX_inv = np.linalg.inv(XTX)

XTX =
[[131950500  60750580]
 [ 60750580  28041424]]
det(XTX) = 9446947175599.648


In [11]:
y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])
w = XTX_inv.dot(X.T).dot(y)
sum_w = float(w.sum())

print("w =", w)
print("sum(w) =", sum_w)
print("Respuesta Q7:", round(sum_w, 3))

w =

 [0.13644777 0.2327492 ]
sum(w) = 0.36919696904925486
Respuesta Q7: 0.369


**¿Por qué `X.dot(X.T)` estaba mal?** `X` es 7×2, así que `X @ X.T` es una matriz
7×7 de **rango ≤ 2** → singular (`det = 0`) y **no invertible**. La ecuación normal de
la regresión lineal usa `XTX = X.T @ X` (aquí 2×2, invertible), no `X @ X.T`.

In [12]:
# Autocomprobacion de las respuestas (Restart & Run All para verificar)
assert n_records == 10000, n_records
assert df["fuel_type"].nunique() == 3
assert n_missing_cols == 2, n_missing_cols
assert max_mpg_asia == 41.2, max_mpg_asia
assert median_before == 254.0 and mode_hp == 252.0 and median_after == 252.0
assert answer_q6 == "Yes, it decreased", answer_q6
assert abs(sum_w - 0.36919696904925486) < 1e-12, sum_w
assert round(sum_w, 3) == 0.369
print("OK: todas las respuestas verificadas")

OK: todas las respuestas verificadas


## Resumen de respuestas

| Pregunta | Respuesta |
|---|---|
| **Q1** — Pandas version | `3.0.6` |
| **Q2** — Records count | `10000` |
| **Q3** — Fuel types | `3` (Gasoline, Diesel, Hybrid) |
| **Q4** — Missing values | `2` columnas: `horsepower` (877) y `acceleration` (264) |
| **Q5** — Max fuel efficiency (Asia) | `41.2` |
| **Q6** — Median of horsepower | **Yes, it decreased** (`254.0` → `252.0`) |
| **Q7** — Sum of weights | `0.369` |

> Q1 depende de tu entorno: aquí se ejecutó con **pandas 3.0.6** (Python 3.14.7).